In [ ]:
import numpy as np
import matplotlib.pyplot as pp
import csv
import pickle

# Objects

In [ ]:
class Layer_Dense:
    def __init__(self, n_inputs, n_neurons):
        self.weights = np.random.randn(n_inputs, n_neurons) * np.sqrt(2. / n_inputs)
        self.biases = np.zeros((1, n_neurons))
        self.weight_momentum = np.zeros_like(self.weights)
        self.bias_momentum = np.zeros_like(self.biases)
    def forward(self, inputs):
        self.inputs = inputs
        dot = np.dot(inputs, self.weights) 
        self.output = dot + self.biases
    def backward(self, dvalues):
        # from NNFS
        self.dweights = np.dot(self.inputs.T, dvalues)
        self.dbiases = np.sum(dvalues, axis=0, keepdims=True)
        self.dinputs = np.dot(dvalues, self.weights.T)


In [ ]:
class ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs = dvalues * (self.inputs > 0)

In [ ]:
class Softmax:
    def forward(self, inputs):
        exp_data = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
        self.output = exp_data / np.sum(exp_data, axis=1, keepdims=True)

In [ ]:
class Loss:
    def forward(self, inputs, targets):
        targets = np.array(targets)
        inputs = np.array(inputs)

        # Clipping inputs to avoid log(0)
        inputs_clipped = np.clip(inputs, 1E-7, 1 - 1E-7)

        if len(targets.shape) == 1:
            return -np.log(inputs_clipped[np.arange(len(inputs_clipped)), targets])
        else:
            i = 0
            input_post_processed = []
            for inputs_1d in inputs_clipped:
                for j, input in enumerate(inputs_1d):
                    if targets[i, j] == 1:
                        input_post_processed.append(input)
                i += 1
            return -np.log(input_post_processed)
    
    def calculate(self, output, input):
        loss_raw_values = self.forward(output, input)
        #self.output = np.mean(loss_raw_values)
        return np.mean(loss_raw_values)
    
    def backward(self, dinputs, targets):
        batch_size = dinputs.shape[0]

        if len(targets.shape) == 1:  
            one_hot_targets = np.zeros_like(dinputs)
            one_hot_targets[np.arange(batch_size), targets] = 1
        else:
            one_hot_targets = targets

        self.dinputs = (dinputs - one_hot_targets) / batch_size
        return self.dinputs
    

In [ ]:
class TrainedModel:
    def __init__(self, layers, activations, loss_function):
        self.layers = layers
        self.activations = activations
        self.loss_function = loss_function

# Using MNIST like Clothing Dataset

In [ ]:
def read_images(file_path):
    labels = []
    images_1D = []
    images_2D = []

    with open(file_path, 'r') as f:
        reader = csv.reader(f)
        # Skips header row
        next(reader)
        for row in reader:
            pixels = np.array(row[1:], dtype=np.uint8)
            labels.append(int(row[0]))
            images_1D.append(pixels)
            images_2D.append(pixels.reshape((28, 28)))

    return labels, images_2D, images_1D


labels_sorted, images2D, images1D_sorted = read_images("data/fashion-mnist_train.csv")

print(len(images1D_sorted))

pp.imshow(images2D[0], cmap='gray')
pp.title("Example image from dataset")
pp.show()

# USING VERTICAL DATA W/ RANDOM OPTIMISATION

In [ ]:
# Input, hidden, output
dims = [784, 64, 32, 10]

# Number of batches and epochs
epoch_count = 50
batch_count = 64

# Learning rate parameters
a_lr = 0.05
s_lr = 0.02
lr = 0
ld = 0.95
min_lr = 0.001
decay_steps = 1000
momentum = 0.09

# Instantiate layers
layers = []
for i in range(len(dims) - 1):
    inp = dims[i]
    out_size = dims[i + 1]
    layers.append(Layer_Dense(inp, out_size))

# Instantiate activations
acts = []
for i in range(len(dims) - 2):
    acts.append(ReLU())
acts.append(Softmax())

# Instantiate loss function
loss_function = Loss()
lowest_loss = 1E+10

# Create copies of the sorted images and labels so they can be shuffled whilst maintaining 
# the origional order on origional arrays
images1D = np.array(images1D_sorted.copy()) / 255.0
labels = np.array(labels_sorted.copy())

# Iterate over epochs
for epoch in range(epoch_count):
    epoch_total_loss, epoch_total_accuracy = 0, 0

    # Set the learning rate for the epoch
    if epoch < 3:
        lr = a_lr
    elif epoch == 3:
        lr = s_lr
    else:
        if epoch % 5 == 0 and epoch != 0:
            if min_lr > lr * ld:
                lr = min_lr
            else:
                lr = lr * ld

    # Make more efficient
    if 0.9 < (0.5 + 0.1 * epoch):
        momentum = 0.9
    else:
        momentum = (0.5 + 0.1 * epoch)

    # Shuffle the images and labels for better training
    idx = np.arange(len(images1D))
    np.random.shuffle(idx)

    images1D = images1D[idx]
    labels = labels[idx]

    # Iterate over batches
    for i in range(0, len(images1D), batch_count):
        
        batch_images = images1D[i:i + batch_count] 
        output = batch_images
        batch_labels = labels[i:i + batch_count]

        # Forward pass through all layers and activations at once (zipped into a tuple to go through both at once)
        for dense, activation in zip(layers, acts):
            dense.forward(output)
            activation.forward(dense.output)
            output = activation.output

        # Calculate loss
        loss = loss_function.calculate(output, batch_labels)
        # Calculate accuracy
        predictions = np.argmax(output, axis=1)
        accuracy = np.mean(predictions == batch_labels)
        epoch_total_loss += loss
        epoch_total_accuracy += accuracy

        # Derivative of the loss wrt final
        loss_function.backward(output, batch_labels)
        doutput = loss_function.dinputs

        # Iterate backwards through all the layers
        for i, (layer_dense, activation) in enumerate(reversed(list(zip(layers, acts)))):
            if i != 0:
                # Skip softmax
                activation.backward(doutput)
                grad_input = activation.dinputs
            else:
                grad_input = doutput

            layer_dense.backward(grad_input)
            doutput = layer_dense.dinputs

            
            # Update weights and biases with momentum   
            layer_dense.weight_momentum = momentum * layer_dense.weight_momentum - lr * layer_dense.dweights
            layer_dense.bias_momentum = momentum * layer_dense.bias_momentum - lr * layer_dense.dbiases

            layer_dense.weights += layer_dense.weight_momentum
            layer_dense.biases += layer_dense.bias_momentum
    

    temp_div = len(images1D) // batch_count

    epoch_loss = epoch_total_loss / temp_div
    epoch_accuracy = epoch_total_accuracy / temp_div

    print(f"Epoch {epoch+1}, LR {lr:.4f} Loss {epoch_loss:.6f}, Accuracy {epoch_accuracy:.4f}")

trained_model = TrainedModel(layers, acts, loss_function)
print("Training complete.")

with open("trained_model.pkl", "wb") as f:
    pickle.dump(trained_model, f)
print("Saved with pickle")

# Test Script

In [ ]:
idx = np.random.randint(0, len(images1D))

test_image = images1D_sorted[idx]
test_image = test_image.reshape(1, -1).astype(np.float32) / 255.0
output = test_image

for dense, activation in zip(trained_model.layers, trained_model.activations):
    dense.forward(output)
    activation.forward(dense.output)
    output = activation.output

prediction = np.argmax(output, axis=1)
confidence = output[0, prediction[0]]

clothing_piece = {
    0: "T-shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}

pp.imshow(images2D[idx], cmap='gray')
pp.title(clothing_piece[prediction[0]] + f" (Confidence: {confidence:.2f})")
pp.show()

# Image Testing

In [ ]:
from PIL import Image, ImageEnhance
import os

input_dir = "assets/example-input.jpeg"
output_dir = "assets/example-input.jpeg"

img = Image.open(input_dir).convert("L")

enhancer_sharp = ImageEnhance.Sharpness(img)
img = enhancer_sharp.enhance(3.0)

enhancer_contrast = ImageEnhance.Contrast(img)
img = enhancer_contrast.enhance(5.0)

resized = img.resize((28,28))

arr = np.array(resized, dtype=np.float32) / 255.0
arr_resized = arr.reshape(1,784)

with open("trained_model.pkl", "rb") as f:
    trained_model = pickle.load(f)
print("Loaded")

output = arr_resized

for dense, activation in zip(trained_model.layers, trained_model.activations):
    dense.forward(output)
    activation.forward(dense.output)
    output = activation.output

prediction = np.argmax(output, axis=1)
confidence = output[0, prediction[0]]

clothing_piece = {
    0: "T-shirt",
    1: "Trouser",
    2: "Pullover",
    3: "Dress",
    4: "Coat",
    5: "Sandal",
    6: "Shirt",
    7: "Sneaker",
    8: "Bag",
    9: "Ankle boot"
}

pp.imshow(arr, cmap='gray')
pp.title(clothing_piece[prediction[0]] + f" (Confidence: {confidence:.2f})")
pp.show()